# 02 — Regras AML, ranking e validação da T1/T2

Objetivo: reutilizar o motor determinístico do case para validar regras explicáveis, rankings e artefatos de investigação.

Este notebook chama diretamente as funções de `src/rules.py`; ele não mantém uma segunda implementação das regras. O motor principal contém 28 regras reproduzíveis: 16 transacionais e 12 cliente-mês.

A R17 de geo-salto permanece demonstrada separadamente nos artefatos da T2 e não é tratada aqui como regra integrada ao motor principal.

In [ ]:
from pathlib import Path
from types import ModuleType
import importlib.util
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
TZ = "America/Sao_Paulo"
np.random.seed(RANDOM_STATE)


def find_project_root(start: Path) -> Path:
    """Localiza a raiz do projeto sem depender do diretório de execução."""
    resolved = start.resolve()

    for candidate in (resolved, *resolved.parents):
        dataset = (
            candidate
            / "data"
            / "raw"
            / "AML_FT_Case_Synthetic_Data.xlsx"
        )

        if dataset.is_file():
            return candidate

    raise FileNotFoundError(
        "Não foi possível localizar "
        "data/raw/AML_FT_Case_Synthetic_Data.xlsx"
    )


def load_rules_module(path: Path) -> ModuleType:
    """Carrega o motor determinístico sem modificar o caminho de imports."""
    specification = importlib.util.spec_from_file_location(
        "case04_rules_notebook",
        path,
    )

    if (
        specification is None
        or specification.loader is None
    ):
        raise ImportError(
            f"Não foi possível carregar o módulo: {path}"
        )

    module = importlib.util.module_from_spec(
        specification
    )
    specification.loader.exec_module(
        module
    )

    return module


ROOT = find_project_root(Path.cwd())
DATA_PATH = (
    ROOT
    / "data"
    / "raw"
    / "AML_FT_Case_Synthetic_Data.xlsx"
)
RULES_PATH = ROOT / "src" / "rules.py"
T1_DIR = ROOT / "outputs" / "t1_suspects"
T2_DIR = ROOT / "outputs" / "t2_alert_system"
OUT_DIR = ROOT / "outputs" / "final_review"

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

rules_module = load_rules_module(
    RULES_PATH
)

print(f"Root: {ROOT}")
print(f"Base existe: {DATA_PATH.exists()} | {DATA_PATH}")
print(f"Motor existe: {RULES_PATH.exists()} | {RULES_PATH}")
print(
    f"T1 existe: {T1_DIR.exists()} | "
    f"T2 existe: {T2_DIR.exists()}"
)

## 1. Execução em memória do motor determinístico

A base sintética é carregada pelo mesmo contrato usado por `src/rules.py`. Os enriquecimentos respeitam `sender_id`, `receiver_id` e os tipos das entidades antes da aplicação das regras.

In [ ]:
if rules_module.XLSX.resolve() != DATA_PATH.resolve():
    raise RuntimeError(
        "O caminho da base no motor diverge do notebook"
    )

dfs = rules_module.load_data()
tx, kyc, merchants, geo = rules_module.prep(dfs)

tx_rules, transaction_catalog = (
    rules_module.add_rules(tx)
)
involved = rules_module.build_inv(
    tx_rules
)
customer_month_alerts, month_catalog = (
    rules_module.month_alerts(
        involved,
        kyc,
    )
)
daily_pass_candidates = (
    rules_module.daily_pass(
        involved
    )
)
customer_ranking = (
    rules_module.rank_clients(
        customer_month_alerts,
        tx_rules,
    )
)

rule_catalog = pd.concat(
    [
        transaction_catalog,
        month_catalog,
    ],
    ignore_index=True,
)

transaction_rule_count = len(
    transaction_catalog
)
month_rule_count = len(
    month_catalog
)
engine_rule_count = len(
    rule_catalog
)

if transaction_rule_count != 16:
    raise RuntimeError(
        "Quantidade inesperada de regras "
        f"transacionais: {transaction_rule_count}"
    )

if month_rule_count != 12:
    raise RuntimeError(
        "Quantidade inesperada de regras "
        f"cliente-mês: {month_rule_count}"
    )

if engine_rule_count != 28:
    raise RuntimeError(
        "Quantidade inesperada de regras "
        f"do motor principal: {engine_rule_count}"
    )

pipeline_summary = pd.DataFrame(
    [
        {
            "metric": "transactions",
            "value": len(tx_rules),
        },
        {
            "metric": "customer_participations",
            "value": len(involved),
        },
        {
            "metric": "customer_month_rows",
            "value": len(customer_month_alerts),
        },
        {
            "metric": "ranked_customers",
            "value": len(customer_ranking),
        },
        {
            "metric": "transaction_rules",
            "value": transaction_rule_count,
        },
        {
            "metric": "customer_month_rules",
            "value": month_rule_count,
        },
        {
            "metric": "principal_engine_rules",
            "value": engine_rule_count,
        },
    ]
)

pipeline_summary.to_csv(
    OUT_DIR / "notebook_02_pipeline_summary.csv",
    index=False,
)

pipeline_summary

## 2. Cobertura das 16 regras transacionais

A cobertura abaixo é calculada diretamente sobre as colunas produzidas por `add_rules`. Pontos, IDs, lógica e thresholds continuam definidos no motor principal, evitando divergência entre notebook e código.

In [ ]:
transaction_rule_cols = (
    transaction_catalog["rule_name"]
    .astype(str)
    .tolist()
)

missing_transaction_rule_cols = sorted(
    set(transaction_rule_cols)
    - set(tx_rules.columns)
)

if missing_transaction_rule_cols:
    raise RuntimeError(
        "Colunas de regras transacionais ausentes: "
        + ", ".join(
            missing_transaction_rule_cols
        )
    )

coverage = transaction_catalog[
    [
        "rule_id",
        "rule_name",
        "level",
        "points",
    ]
].copy()

coverage["trigger_count"] = [
    int(tx_rules[column].sum())
    for column in transaction_rule_cols
]

coverage = coverage.sort_values(
    [
        "trigger_count",
        "points",
        "rule_id",
    ],
    ascending=[
        False,
        False,
        True,
    ],
).reset_index(
    drop=True
)

coverage.to_csv(
    OUT_DIR
    / "notebook_02_transaction_rule_coverage.csv",
    index=False,
)

coverage

## 3. Ranking transacional do motor principal

A fila abaixo usa `tx_rule_count` e `tx_rule_score` produzidos pelo motor. Concentração de sinais e severidade orientam a priorização, mas a decisão permanece sujeita à revisão humana.

In [ ]:
transaction_columns = [
    "transaction_id",
    "subject_customer_id",
    "timestamp",
    "transaction_type",
    "status",
    "amount_brl",
    "tx_rule_count",
    "tx_rule_score",
    "tx_rules_triggered",
    "kyc_risk_rating",
    "kyc_pep",
    "country_risk_receiver",
    "merchant_mcc_risk",
    "sanctions_screening_hit",
    "cross_border",
    "ip_anomaly",
    "device_rooted",
]

transaction_columns = [
    column
    for column in transaction_columns
    if column in tx_rules.columns
]

top_tx_demo = (
    tx_rules.loc[
        tx_rules["tx_rule_count"].gt(0)
    ]
    .sort_values(
        [
            "tx_rule_score",
            "amount_brl",
            "transaction_id",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    [transaction_columns]
    .head(30)
)

top_tx_demo.to_csv(
    OUT_DIR
    / "notebook_02_demo_top30_transactions.csv",
    index=False,
)

top_tx_demo.head(10)

## 4. Cobertura das 12 regras cliente-mês

As tipologias agregadas são produzidas por `month_alerts`, incluindo fora de perfil com threshold dinâmico, velocity, structuring, pass-through, concentração cross-border, sanções, PEP e recorrência de sinais.

In [ ]:
month_rule_cols = (
    month_catalog["rule_name"]
    .astype(str)
    .tolist()
)

missing_month_rule_cols = sorted(
    set(month_rule_cols)
    - set(customer_month_alerts.columns)
)

if missing_month_rule_cols:
    raise RuntimeError(
        "Colunas de regras cliente-mês ausentes: "
        + ", ".join(
            missing_month_rule_cols
        )
    )

month_coverage = month_catalog[
    [
        "rule_id",
        "rule_name",
        "level",
        "points",
    ]
].copy()

month_coverage["trigger_count"] = [
    int(
        customer_month_alerts[column].sum()
    )
    for column in month_rule_cols
]

month_coverage = month_coverage.sort_values(
    [
        "trigger_count",
        "points",
        "rule_id",
    ],
    ascending=[
        False,
        False,
        True,
    ],
).reset_index(
    drop=True
)

top_customer_month = (
    customer_month_alerts
    .sort_values(
        [
            "month_rule_score",
            "total_amount",
            "customer_id",
            "month",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ],
    )
    .head(30)
)

top_clients_demo = (
    customer_ranking
    .head(30)
    .copy()
)

month_coverage.to_csv(
    OUT_DIR
    / "notebook_02_customer_month_rule_coverage.csv",
    index=False,
)

top_customer_month.to_csv(
    OUT_DIR
    / "notebook_02_demo_top30_customer_month.csv",
    index=False,
)

top_clients_demo.to_csv(
    OUT_DIR
    / "notebook_02_demo_top30_clients.csv",
    index=False,
)

top_customer_month[
    [
        column
        for column in [
            "customer_id",
            "month",
            "month_rule_count",
            "month_rule_score",
            "total_amount",
            "tx_count",
            "month_rules_triggered",
        ]
        if column in top_customer_month.columns
    ]
].head(10)

## 5. Validação dos artefatos T1 e T2

Esta etapa verifica os arquivos usados na investigação e compara seus catálogos com o motor executado em memória. O SAR é localizado por padrão de arquivo, sem fixar um cliente específico.

O catálogo da T2 deve conter as 28 regras do motor e a R17 de geo-salto como evidência separada, ainda não integrada ao fluxo reproduzível do motor principal.

In [ ]:
sar_drafts = sorted(
    T1_DIR.glob(
        "07_SAR_draft_*.md"
    )
)

required_outputs = [
    T1_DIR / "01_rule_catalog_t1.csv",
    T1_DIR / "02_suspicious_transactions_top30.csv",
    T1_DIR / "03_suspicious_clients_top30.csv",
    T1_DIR / "04_client_month_alerts_all.csv",
    T1_DIR / "05_daily_pass_through_candidates.csv",
    T2_DIR / "01_alert_rules_catalog_t2.csv",
    T2_DIR / "04b_geo_jump_summary.json",
]

validation_rows = [
    {
        "artifact": path.relative_to(ROOT).as_posix(),
        "exists": path.is_file(),
        "size_bytes": (
            path.stat().st_size
            if path.is_file()
            else 0
        ),
    }
    for path in required_outputs
]

if sar_drafts:
    validation_rows.extend(
        {
            "artifact": path.relative_to(ROOT).as_posix(),
            "exists": True,
            "size_bytes": path.stat().st_size,
        }
        for path in sar_drafts
    )
else:
    validation_rows.append(
        {
            "artifact": (
                "outputs/t1_suspects/"
                "07_SAR_draft_*.md"
            ),
            "exists": False,
            "size_bytes": 0,
        }
    )

validation = pd.DataFrame(
    validation_rows
)

validation.to_csv(
    OUT_DIR
    / "notebook_02_required_outputs_validation.csv",
    index=False,
)

validation

In [ ]:
missing_artifacts = validation.loc[
    ~validation["exists"],
    "artifact",
].tolist()

if missing_artifacts:
    raise FileNotFoundError(
        "Artefatos obrigatórios ausentes: "
        + ", ".join(
            missing_artifacts
        )
    )

final_top_tx = pd.read_csv(
    T1_DIR
    / "02_suspicious_transactions_top30.csv"
)
final_top_clients = pd.read_csv(
    T1_DIR
    / "03_suspicious_clients_top30.csv"
)
t1_catalog = pd.read_csv(
    T1_DIR
    / "01_rule_catalog_t1.csv"
)
t2_catalog = pd.read_csv(
    T2_DIR
    / "01_alert_rules_catalog_t2.csv"
)

engine_rule_ids = set(
    rule_catalog["rule_id"].astype(str)
)
t1_rule_ids = set(
    t1_catalog["rule_id"].astype(str)
)
t2_rule_ids = set(
    t2_catalog["rule_id"].astype(str)
)

if len(engine_rule_ids) != 28:
    raise RuntimeError(
        "O motor em memória não possui "
        "28 IDs de regra únicos"
    )

if "R17" in engine_rule_ids:
    raise RuntimeError(
        "A R17 não deve estar integrada "
        "ao motor principal nesta etapa"
    )

if t1_rule_ids != engine_rule_ids:
    raise RuntimeError(
        "O catálogo T1 diverge "
        "do motor principal"
    )

t2_missing_engine_ids = sorted(
    engine_rule_ids
    - t2_rule_ids
)
t2_extra_rule_ids = sorted(
    t2_rule_ids
    - engine_rule_ids
)

if t2_missing_engine_ids:
    raise RuntimeError(
        "O catálogo T2 não cobre regras "
        "do motor principal: "
        + ", ".join(
            t2_missing_engine_ids
        )
    )

if t2_extra_rule_ids != ["R17"]:
    raise RuntimeError(
        "Diferença inesperada entre "
        "o catálogo T2 e o motor: "
        + ", ".join(
            t2_extra_rule_ids
        )
    )

sar_draft_path = sar_drafts[0]

catalog_comparison = pd.DataFrame(
    [
        {
            "contract": "transaction_rules",
            "value": transaction_rule_count,
            "status": "PASS",
        },
        {
            "contract": "customer_month_rules",
            "value": month_rule_count,
            "status": "PASS",
        },
        {
            "contract": "principal_engine_rules",
            "value": len(engine_rule_ids),
            "status": "PASS",
        },
        {
            "contract": "t1_catalog_matches_engine",
            "value": len(t1_rule_ids),
            "status": "PASS",
        },
        {
            "contract": "r17_in_principal_engine",
            "value": False,
            "status": "PASS",
        },
        {
            "contract": "r17_in_t2_catalog",
            "value": "R17" in t2_rule_ids,
            "status": "PASS",
        },
        {
            "contract": "r17_integration_status",
            "value": "separate_not_integrated",
            "status": "PASS",
        },
    ]
)

catalog_comparison.to_csv(
    OUT_DIR
    / "notebook_02_rule_catalog_comparison.csv",
    index=False,
)

print(
    "Top transações finais:",
    final_top_tx.shape,
)
print(
    "Top clientes finais:",
    final_top_clients.shape,
)
print(
    "Regras no motor principal:",
    len(engine_rule_ids),
)
print(
    "Regras no catálogo T2:",
    len(t2_rule_ids),
)
print(
    "SAR localizado:",
    sar_draft_path.relative_to(ROOT),
)

catalog_comparison

In [ ]:
chart_data = (
    coverage
    .head(16)
    .sort_values(
        "trigger_count"
    )
)

plt.figure(
    figsize=(9, 6)
)

chart_data.plot(
    kind="barh",
    x="rule_name",
    y="trigger_count",
    legend=False,
    ax=plt.gca(),
)

plt.title(
    "Cobertura das 16 regras transacionais "
    "do motor principal"
)
plt.xlabel(
    "Quantidade de acionamentos"
)
plt.ylabel(
    "Regra"
)
plt.tight_layout()

plt.savefig(
    OUT_DIR
    / "notebook_02_rule_coverage.png",
    dpi=160,
)

plt.show()

## 6. Conclusão

O notebook reutiliza o núcleo determinístico do case e evita uma implementação paralela das regras. A fila de investigação combina evidências transacionais e cliente-mês, com rankings explicáveis e revisão humana obrigatória.

O motor principal permanece composto por 28 regras reproduzíveis. A R17 de geo-salto é validada como artefato separado da T2 e não deve ser apresentada como integrada ao motor enquanto essa integração não for implementada e testada ponta a ponta.